In [1]:
from datasets import load_dataset, load_from_disk
import polars as pl
import pandas as pd
import json
import pickle
import numpy as np
from replay.metrics import Recall, Precision, HitRate
import faiss
from functools import reduce
import datasets
import torch
from tqdm import tqdm
import os
from datetime import datetime

pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_rows(-1)

DATA_PATH = "/home/jupyter/filestore/storage/datasets"

In [2]:
dataset = load_from_disk(f"{DATA_PATH}/user_events_20230501")
polars_ds = dataset.to_polars()

In [5]:
index = faiss.read_index("data/neural_index.faiss")

with open("data/item_ids", "rb") as fp:
    item_ids = pickle.load(fp)
    
item_embs = np.load("data/item_embeddings.npy")
itemid2idx = {item_id:idx for idx, item_id in enumerate(item_ids)}

In [38]:
replace_func = np.vectorize(lambda x: item_ids[x])

queries = item_embs[[itemid2idx[137954989]]]
_, idx = index.search(queries, k=20)
recs = replace_func(idx)

# Примеры рекомендаций похожих товаров

In [13]:
polars_ds.filter(pl.col("item_id") == 132006685).unique().select("item_id", "name")

item_id,name
i64,str
132006685,"""Hot Wheels Premium Jay Leno's Garage McLaren F1"""


In [15]:
polars_ds.select("name", "item_id").filter(pl.col("item_id").is_in(recs[0].tolist())).unique().head(10)

name,item_id
str,i64
"""Hot Wheels Premium Speed Machines Porsche 911 GT3 1/64""",127528389
"""2009 Hot Wheels""",144408552
"""Hot Wheels Premium Jay Leno's Garage McLaren F1""",132006685
"""Hot Wheels Jay Leno’s Garage McLAREN F1 Pair""",177577493
"""Hot Wheels 83 Chevy Silverado Toon’d set""",234407955
"""Hot Wheels Premium Speed Machines Porsches 911 GT3""",228876500
"""Hot Wheels Yellow Exclusive McLaren""",165235474
"""Hot Wheels Ultra Hots Porsche 934.5""",160251496
"""2015 Hot Wheels Stars & Stripes ‘70 Ford Torino""",68116108


In [18]:
polars_ds.filter(pl.col("item_id") == 236809122).unique().select("item_id", "name")

item_id,name
i64,str
236809122,"""Juice WRLD vinyl bundle"""


In [22]:
polars_ds.select("name", "item_id").filter(pl.col("item_id").is_in(recs[0].tolist())).unique().head(10)

name,item_id
str,i64
"""Juice Wrld Goodbye & Good Riddance Vinyl""",156263864
"""Limp Bizkit Vinyl Results May Vary Green Vinyl 2LP""",222200671
"""The weeknd RARE splatter heavy weight 180 gram colored vinyl record after hours""",48574175
"""Juice wrld goodbye and good riddance blue vinyl""",53615677
"""The Weeknd - Thursday - Vinyl 2LP - NEW SEALED""",142983806
"""Mac Miller - Blue Slide Park 2LP 10th Anniversary Deluxe Vinyl (COLOR VINYL)""",161918114
"""Juice WRLD vinyl bundle""",236809122
"""Juice wrld vinyl""",227295500
"""Juice Wrld Goodbye Good Riddance Vinyl""",222916388


In [20]:
polars_ds.filter(pl.col("item_id") == 127421407).unique().select("item_id", "name")

item_id,name
i64,str
127421407,"""Polo Ralph Lauren Jeans Classic 867 Straight Leg"""


In [24]:
polars_ds.select("name", "item_id").filter(pl.col("item_id").is_in(recs[0].tolist())).unique().head(10)

name,item_id
str,i64
"""Mens Denim Supply Ralph Lauren jeans 34""",69255710
"""Lauren Ralph Lauren total comfort trouser mens 34x32 black office pants""",236408463
"""Skinny Jeans""",197133036
"""Calvin Klein jeans straight leg 34/""",174164446
"""Polo Ralph Lauren pockets cargo pants""",161864972
"""Polo Classic Fit Men's Jeans""",21895768
"""Vintage polo Ralph Lauren loose fit jeans 34x30!""",2132179
"""Polo Ralph Lauren Pants""",38271137
"""Polo Ralph Lauren Vintage Jeans""",204202888


In [37]:
polars_ds.filter(pl.col("item_id") == 137954989).unique()[0].select("item_id", "name")

item_id,name
i64,str
137954989,"""Michael Kors Keaton Slip On Sneakers"""


In [39]:
polars_ds.select("name", "item_id").filter(pl.col("item_id").is_in(recs[0].tolist())).unique().head(10)

name,item_id
str,i64
"""Michael Kors Nikko High-Top Wedge Sneake""",136092799
"""WOMENS RICK OWEN DRKSHDW""",158058776
"""Michael Kors Bootie""",77246405
"""New Michael kors Slingback heels red/gold various sizes""",73289746
"""Michael Kors bootie""",132373553
"""Michael Kors rain boots""",177855824
"""Kurt keiger leather sneakers""",60115547
"""Jimmy Choo Edina Etched Cork Heels Shoes""",114793117
"""Vans SK8-Hi Freddy Krueger shoes""",104810758
